In [1]:
from py_module.metadata.connection.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.render_jinja.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from py_module.metadata.connection.StorageBaseConnection import StorageConn as storage
from py_module.metadata.datatype_conversion.avro import DataTypeConverter as dtc
from sqlalchemy import text
import json
from fastavro import writer, parse_schema
from datetime import datetime as dt


# SETUP (connection + get template)

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

In [4]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

# Render template with appropriate values

### Retrieve schema table -- this is useful for next steps

In [5]:
table_schema = 'public'
table_name = 'fct_meteo'
rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )

In [6]:
it_schema = conn.execute(text(rendered_schema))
converter = dtc()  # your DataTypeConverter instance, optionally pass defaults

dbt_columns = list()
avro_columns = list()
cdc_columns = list()

for i in it_schema:
    col_name = i[0]
    db_type = i[1]
    precision = i[2]
    scale = i[3]

    avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
    bq_type = converter.source_to_bigquery(params['database'], db_type)

    # setup for cdc model injection
    cdc_columns.append(col_name)

    # setup the columns for avro injection with default
    avro_field = converter.generate_avro_field(col_name, avro_type)
    avro_columns.append(avro_field)

    # setup the columns for dbt source
    dbt_columns.append({col_name: bq_type})


In [7]:

rendered_sources = schema_source.render(
    schema_name = table_schema,
    table_name = table_name,
    cols = dbt_columns,
    database = params['database'],
    staging_dataset = 'staging',
    istance_name = params['db'],
    bucket_name = 'postgres__d-meteo-db',
    version = 'v1'
    )

### define custom CDC extraction query 

In [8]:
rendered_cdc = cdc_extraction.render(
                                columns = cdc_columns,
                                schema_name = table_schema, 
                                table_name = table_name, 
                                delta = False, 
                                # delta_column = delta_column, 
                                # delta_timestamp = delta_timestamp, 
                                # where_conditions = where_conditions
                            )

In [9]:
avro_schema = {
    "name": f"{table_schema}_{table_name}__record",
    "type": "record",
    "fields": avro_columns,
}
parsed_schema = parse_schema(avro_schema)

In [10]:
cdc_exec = conn.execution_options(stream_results=True).execute(text(rendered_cdc))

cols = cdc_exec.keys()
today = dt.today().strftime("%Y-%m-%d")
now = dt.now().strftime("%d%m%Y%H%M%S")
gcs = storage()

blob = gcs.define_blob(
    bucket_name='postgres__d-meteo-db',
    blob_name=f'{table_schema}/{table_name}/ingestion_date={today}/chunk__{now}.avro'
)

In [13]:
from decimal import Decimal, ROUND_DOWN

# Map decimal fields to their scale from schema
decimal_fields = {}
for field in parsed_schema['fields']:
    f_type = field['type']
    if isinstance(f_type, dict) and f_type.get('logicalType') == 'decimal':
        decimal_fields[field['name']] = f_type['scale']
    # handle nullable decimals
    elif isinstance(f_type, list):
        for t in f_type:
            if isinstance(t, dict) and t.get('logicalType') == 'decimal':
                decimal_fields[field['name']] = t['scale']

def fix_decimal(value, scale):
    """Ensure value is a Decimal with correct scale."""
    if value is None:
        return None
    # convert float or string to Decimal
    d = Decimal(value)
    quantize_str = '0.' + '0' * (scale - 1) + '1'  # e.g., "0.01" for scale 2
    return d.quantize(Decimal(quantize_str), rounding=ROUND_DOWN)

def record_generator():
    for row in cdc_exec:
        record = dict(zip(cols, row))
        for field, scale in decimal_fields.items():
            if field in record and record[field] is not None:
                record[field] = fix_decimal(record[field], scale)
        yield record


In [15]:
with blob.open('wb', ignore_flush=True) as f:
    avro_writer = writer(f, parsed_schema, records = record_generator())

TypeError: writer() takes at least 3 positional arguments (2 given)

In [ ]:

    # Stream rows and write one-by-one
    for row in result:
        record = {}
        for col_name, value in zip(cols, row):
            # If your schema expects non-nullable fields, handle nulls here if needed
            record[col_name] = value
        avro_writer.write(record)

[{'name': 'observation_pk', 'type': 'text'},
 {'name': 'region', 'type': 'text'},
 {'name': 'province', 'type': 'text'},
 {'name': 'province_code', 'type': 'text'},
 {'name': 'city', 'type': 'text'},
 {'name': 'observation_dt', 'type': 'timestamp'},
 {'name': 'summary', 'type': 'text'},
 {'name': 'precip_intensity', 'type': 'numeric'},
 {'name': 'precip_accumulation', 'type': 'numeric'},
 {'name': 'precip_type', 'type': 'text'},
 {'name': 'temperature', 'type': 'numeric'},
 {'name': 'apparent_temperature', 'type': 'numeric'},
 {'name': 'dew_point', 'type': 'numeric'},
 {'name': 'pressure', 'type': 'numeric'},
 {'name': 'wind_speed', 'type': 'numeric'},
 {'name': 'wind_gust', 'type': 'numeric'},
 {'name': 'windb_earing', 'type': 'numeric'},
 {'name': 'cloud_cover', 'type': 'numeric'},
 {'name': 'snow_accumulation', 'type': 'numeric'},
 {'name': 'insert_timestamp', 'type': 'timestamp'},
 {'name': 'update_timestamp', 'type': 'timestamp'}]